# Experiment 2: Upworthy Zero-Shot Classification Threshold Optimization

**STAT 238 Final Project — Bayesian Optimization for ML Pipelines**

---

## What is this experiment?

We have 55,092 Upworthy headline A/B test records. Each headline was classified
into one of 7 topic categories (politics, health, environment, etc.) using
[`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli),
a zero-shot NLI model. BART gives both a **label** and a **confidence score** (0–1).

The question: **how should we configure the classification pipeline so the
resulting categories are maximally predictive of click-through rate (CTR)?**

### Two configuration knobs

| Parameter | Range | Meaning |
|-----------|-------|---------|
| `confidence_threshold` | [0.05, 0.90] | Headlines below this score are labelled "other" and excluded from analysis |
| `min_cat_size` | [10, 400] | Minimum headlines per category required to include that group in the ANOVA |

### Why is this non-trivial?

- **Threshold too low** → BART's uncertain guesses pollute categories → noisy group means → low F-statistic
- **Threshold too high** → very few headlines get labeled → tiny groups → degenerate or high-variance ANOVA
- **These two parameters interact**: stricter threshold → smaller groups → the minimum size matters more

### Objective: one-way ANOVA F-statistic on log(CTR)

$$F = \frac{\text{variance in } \log(\text{CTR}) \textit{ between categories}}{\text{variance in } \log(\text{CTR}) \textit{ within categories}}$$

High $F$ means topic category is predictive of CTR — the classification is meaningful.
We **maximise** $F$ over the 2D parameter space using Bayesian Optimisation.

### Why Bayesian Optimisation?

The mapping $(\text{threshold},\, \text{min\_cat}) \mapsto F$ has no closed form.
Grid search is inefficient in 2D (exponential in the number of dimensions).
BO uses a GP surrogate to model the objective landscape and EI to decide
where to evaluate next — concentrating evaluations near the optimum.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import f_oneway

REPO_ROOT = Path("..")
sys.path.insert(0, str(REPO_ROOT))

from src.gp import GaussianProcess
from src.acquisition import expected_improvement, next_best_candidate
from src.black_box_upworthy import UPWORTHY_PARAM_SPACE, decode_params

# --- Paths ---
RESULTS_DIR   = REPO_ROOT / "results" / "upworthy"
FIGURES_DIR   = REPO_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

UPWORTHY_REPO = Path(
    "/Users/ricardoperezcastillo/Library/CloudStorage/"
    "OneDrive-Personal/Documents/Berkeley/Spring 2026/"
    "STAT 230A/Final Project/Github/A-B-testing-analysis-upworthy"
)

# --- Plot style ---
plt.rcParams.update({
    "figure.dpi":      120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size":       11,
})

print("Setup complete.")

## 1  Load BO, Random Search, and Grid Search Results

In [ ]:
bo_df     = pd.read_csv(RESULTS_DIR / "bo_trials.csv")
rand_df   = pd.read_csv(RESULTS_DIR / "random_trials.csv")
grid_df   = pd.read_csv(RESULTS_DIR / "grid_trials.csv")

# Add best-so-far column (cumulative max of f_stat)
bo_df["best_so_far"]   = bo_df["f_stat"].cummax()
rand_df["best_so_far"] = rand_df["f_stat"].cummax()
grid_df["best_so_far"] = grid_df["f_stat"].cummax()

# Phase labels for BO: first 10 are random init, rest are BO
N_INIT = 10
bo_df["phase"] = np.where(bo_df["trial"] <= N_INIT, "random", "BO")

print(f"BO trials       : {len(bo_df)}")
print(f"Random trials   : {len(rand_df)}")
print(f"Grid trials     : {len(grid_df)}")
print()
print(f"BO best F       : {bo_df['f_stat'].max():.2f}  "
      f"(threshold={bo_df.loc[bo_df['f_stat'].idxmax(), 'confidence_threshold']:.3f}, "
      f"min_cat={bo_df.loc[bo_df['f_stat'].idxmax(), 'min_cat_size']})")
print(f"Random best F   : {rand_df['f_stat'].max():.2f}")
print(f"Grid best F     : {grid_df['f_stat'].max():.2f}")

## 2  The Objective Landscape

Before looking at BO, let's understand the objective. We plot the F-statistic
as a function of `confidence_threshold` (holding `min_cat_size` ≈ 86, the BO optimum)
to see why this isn't trivial.

In [ ]:
# Reconstruct the 1D slice from BO trials closest to min_cat_size ≈ 86
# (use all trials sorted by threshold)
slice_df = bo_df.sort_values("confidence_threshold").copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Left: F-stat vs threshold (scatter all trials) ---
ax = axes[0]
bo_rand   = bo_df[bo_df["phase"] == "random"]
bo_bo     = bo_df[bo_df["phase"] == "BO"]

ax.scatter(bo_rand["confidence_threshold"], bo_rand["f_stat"],
           c="steelblue", alpha=0.7, s=40, label="Random init", zorder=3)
ax.scatter(bo_bo["confidence_threshold"], bo_bo["f_stat"],
           c="darkorange", alpha=0.7, s=40, label="BO trial", zorder=3)

best_row = bo_df.loc[bo_df["f_stat"].idxmax()]
ax.axvline(best_row["confidence_threshold"], color="crimson", lw=1.5,
           ls="--", label=f"Best (thresh={best_row['confidence_threshold']:.3f})")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("ANOVA F-statistic")
ax.set_title("F-statistic vs confidence threshold\n(all BO trials)")
ax.legend(fontsize=9)

# --- Right: coverage vs threshold ---
ax = axes[1]
ax.scatter(bo_df["confidence_threshold"], bo_df["coverage"] * 100,
           c="mediumpurple", alpha=0.7, s=40)
ax.axvline(best_row["confidence_threshold"], color="crimson", lw=1.5,
           ls="--", label=f"Best threshold")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("% headlines labeled (coverage)")
ax.set_title("Coverage drops as threshold increases\n(label → 'other' if confidence < threshold)")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "upworthy_objective_landscape.png", bbox_inches="tight")
plt.show()
print("Saved: upworthy_objective_landscape.png")

## 3  Convergence: BO vs Random Search vs Grid Search

The convergence curve shows how quickly each method finds a good configuration.
BO should improve faster because it uses past evaluations to guide the search.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(bo_df["trial"],   bo_df["best_so_far"],
        color="darkorange", lw=2.5, label="Bayesian Optimisation", zorder=3)
ax.plot(rand_df["trial"], rand_df["best_so_far"],
        color="steelblue",  lw=2.0, ls="--", label="Random search")
ax.plot(grid_df["trial"], grid_df["best_so_far"],
        color="seagreen",   lw=2.0, ls=":",  label="Grid search")

# Mark where BO switches from random init to EI-guided
ax.axvline(N_INIT, color="gray", lw=1, ls="-.", alpha=0.7)
ax.text(N_INIT + 0.5, ax.get_ylim()[0] * 1.01,
        "← random  BO →", va="bottom", ha="left",
        color="gray", fontsize=9)

# Annotate final values
for df, color, name in [
    (bo_df,   "darkorange", "BO"),
    (rand_df, "steelblue",  "Random"),
    (grid_df, "seagreen",   "Grid"),
]:
    best = df["best_so_far"].iloc[-1]
    ax.annotate(f"{name}: F={best:.0f}",
                xy=(df["trial"].iloc[-1], best),
                xytext=(5, 0), textcoords="offset points",
                color=color, fontsize=9, va="center")

ax.set_xlabel("Trial number")
ax.set_ylabel("Best ANOVA F-statistic found so far")
ax.set_title("Convergence: Bayesian Optimisation vs Baselines\nUpworthy classification threshold tuning")
ax.legend(loc="lower right")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "upworthy_convergence.png", bbox_inches="tight")
plt.show()
print("Saved: upworthy_convergence.png")

## 4  GP Posterior: What BO Learned About the Objective

After running all 60 trials, we condition the GP on the observed data and plot
the **posterior mean** — the GP's belief about the F-statistic landscape over
the full 2D parameter space.

The GP inputs are normalised to $[0,1]^2$ (as in `bo.py`) and outputs are
standardised. We un-standardise for plotting in original F-stat units.

In [ ]:
import math

# --- Rebuild GP from saved trials ---

# Param bounds (original scale)
thresh_lo, thresh_hi   = UPWORTHY_PARAM_SPACE[0]["bounds"]
log_mcs_lo, log_mcs_hi = UPWORTHY_PARAM_SPACE[1]["bounds"]

# Build X matrix in original scale
X_raw = bo_df[["confidence_threshold", "log_min_cat_size"]].values  # (60, 2)
y_raw = bo_df["f_stat"].values                                        # (60,)

# Normalise to [0,1]^2  (mirrors bo.py _to_unit)
lo = np.array([thresh_lo, log_mcs_lo])
hi = np.array([thresh_hi, log_mcs_hi])
X_unit = (X_raw - lo) / (hi - lo)

# Standardise y  (mirrors bo.py _standardise_y)
y_mu    = float(np.mean(y_raw))
y_sigma = float(np.std(y_raw))
y_std   = (y_raw - y_mu) / y_sigma

# Condition GP
gp = GaussianProcess(n_restarts=5)
gp.condition(X_unit, y_std)
print(f"Fitted GP: {gp}")

# --- Prediction grid ---
N_GRID = 80
thresh_vals   = np.linspace(thresh_lo,   thresh_hi,   N_GRID)
log_mcs_vals  = np.linspace(log_mcs_lo, log_mcs_hi, N_GRID)
TT, LL = np.meshgrid(thresh_vals, log_mcs_vals)   # (N_GRID, N_GRID)

# Stack into (N_GRID^2, 2) and normalise
X_grid_raw  = np.column_stack([TT.ravel(), LL.ravel()])
X_grid_unit = (X_grid_raw - lo) / (hi - lo)

mu_std, var_std = gp.predict(X_grid_unit)

# Un-standardise: mu in original F-stat units
mu_orig = mu_std * y_sigma + y_mu

# Axis for min_cat_size ticks (decode log values)
log_mcs_ticks = np.log([10, 20, 50, 100, 200, 400])
mcs_tick_labels = ["10", "20", "50", "100", "200", "400"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: GP posterior mean ---
ax = axes[0]
cf = ax.contourf(
    thresh_vals, log_mcs_vals, mu_orig.reshape(N_GRID, N_GRID),
    levels=30, cmap="viridis"
)
fig.colorbar(cf, ax=ax, label="Predicted F-statistic")
ax.scatter(X_raw[:N_INIT, 0], X_raw[:N_INIT, 1],
           c="white", edgecolors="k", s=40, lw=0.7,
           label="Random init", zorder=5)
ax.scatter(X_raw[N_INIT:, 0], X_raw[N_INIT:, 1],
           c="red", edgecolors="k", s=40, lw=0.7,
           label="BO trial", zorder=5)
# Mark best
best_idx = int(np.argmax(y_raw))
ax.scatter(X_raw[best_idx, 0], X_raw[best_idx, 1],
           c="yellow", edgecolors="k", s=120, lw=1.2,
           marker="*", label="Best found", zorder=6)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Min category size (log scale)")
ax.set_yticks(log_mcs_ticks)
ax.set_yticklabels(mcs_tick_labels)
ax.set_title("GP Posterior Mean\n(estimated F-statistic landscape)")
ax.legend(fontsize=9)

# --- Right: GP posterior std (uncertainty) ---
ax = axes[1]
std_orig = np.sqrt(np.clip(var_std, 0, None)) * y_sigma
cf2 = ax.contourf(
    thresh_vals, log_mcs_vals, std_orig.reshape(N_GRID, N_GRID),
    levels=20, cmap="plasma"
)
fig.colorbar(cf2, ax=ax, label="Posterior std (F-stat units)")
ax.scatter(X_raw[:N_INIT, 0], X_raw[:N_INIT, 1],
           c="white", edgecolors="k", s=40, lw=0.7, label="Random init", zorder=5)
ax.scatter(X_raw[N_INIT:, 0], X_raw[N_INIT:, 1],
           c="red", edgecolors="k", s=40, lw=0.7, label="BO trial", zorder=5)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Min category size (log scale)")
ax.set_yticks(log_mcs_ticks)
ax.set_yticklabels(mcs_tick_labels)
ax.set_title("GP Posterior Uncertainty\n(high = unexplored region)")
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "upworthy_gp_posterior.png", bbox_inches="tight")
plt.show()
print("Saved: upworthy_gp_posterior.png")

## 5  EI Surface: What the Acquisition Function Saw

The Expected Improvement surface shows what BO was "thinking" after all 60 trials:
where in the parameter space does the GP predict unexplored points *might* beat
the current best? Low EI everywhere = BO is confident it has found the optimum.

In [ ]:
# EI in normalised space with standardised y_best
y_best_std = float(y_std.max())
ei_vals = expected_improvement(X_grid_unit, gp, y_best=y_best_std, xi=0.0)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(
    thresh_vals, log_mcs_vals, ei_vals.reshape(N_GRID, N_GRID),
    levels=25, cmap="YlOrRd"
)
fig.colorbar(cf, ax=ax, label="Expected Improvement (standardised)")
ax.scatter(X_raw[:N_INIT, 0], X_raw[:N_INIT, 1],
           c="white", edgecolors="k", s=40, lw=0.7, label="Random init", zorder=5)
ax.scatter(X_raw[N_INIT:, 0], X_raw[N_INIT:, 1],
           c="royalblue", edgecolors="k", s=40, lw=0.7, label="BO trial", zorder=5)
ax.scatter(X_raw[best_idx, 0], X_raw[best_idx, 1],
           c="yellow", edgecolors="k", s=150, lw=1.2,
           marker="*", label="Best found", zorder=6)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Min category size (log scale)")
ax.set_yticks(log_mcs_ticks)
ax.set_yticklabels(mcs_tick_labels)
ax.set_title("Expected Improvement surface after 60 trials\n(low everywhere = BO converged)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "upworthy_ei_surface.png", bbox_inches="tight")
plt.show()
print("Saved: upworthy_ei_surface.png")

## 6  Best Configuration: Interpreting the Result

The BO optimum tells us the best way to post-process BART's predictions.
Let's look at what that configuration actually does to the data.

In [ ]:
# Load the data (mirrors UpworthyBlackBox.__init__)
cats  = pd.read_csv(UPWORTHY_REPO / "data/processed/categories_raw.csv")
clean = pd.read_csv(UPWORTHY_REPO / "data/processed/confirmatory_clean.csv",
                    usecols=["headline", "log_ctr"])
df = clean.merge(cats, on="headline", how="inner")

# Best config
best_row  = bo_df.loc[bo_df["f_stat"].idxmax()]
BEST_THRESH = float(best_row["confidence_threshold"])
BEST_MINCAT = int(best_row["min_cat_size"])
BEST_F      = float(best_row["f_stat"])

print(f"Best configuration found by BO:")
print(f"  confidence_threshold = {BEST_THRESH:.3f}")
print(f"  min_cat_size         = {BEST_MINCAT}")
print(f"  F-statistic          = {BEST_F:.2f}")
print(f"  coverage             = {float(best_row['coverage']):.1%}")
print()

# Apply best config
df["category"] = df["category_raw"].where(
    df["category_confidence"] >= BEST_THRESH, "other"
)
groups_best = {
    cat: grp["log_ctr"].values
    for cat, grp in df[df["category"] != "other"].groupby("category_raw")
    if len(grp) >= BEST_MINCAT
}

# Paper default (threshold=0.30)
PAPER_THRESH = 0.30
PAPER_MINCAT = 1
groups_paper = {
    cat: grp["log_ctr"].values
    for cat, grp in df[df["category_confidence"] >= PAPER_THRESH].groupby("category_raw")
    if len(grp) >= PAPER_MINCAT
}
f_paper, _ = f_oneway(*groups_paper.values())

print(f"Paper default (threshold=0.30, min_cat=1): F = {f_paper:.2f}")
print(f"BO optimum   (threshold={BEST_THRESH:.3f}, min_cat={BEST_MINCAT}): F = {BEST_F:.2f}")
print(f"Improvement: +{(BEST_F/f_paper - 1)*100:.1f}%")

In [ ]:
# Box plots: log(CTR) by category under best configuration
cat_order = sorted(groups_best, key=lambda c: np.median(groups_best[c]), reverse=True)
data_ordered = [groups_best[c] for c in cat_order]

# Shorten labels for display
label_map = {
    "politics and government":       "Politics/Gov't",
    "social justice and inequality": "Social Justice",
    "health and medicine":           "Health",
    "environment and climate":       "Environment",
    "science and technology":        "Science/Tech",
    "economics and labor":           "Economics",
    "lifestyle and culture":         "Lifestyle",
}
short_labels = [label_map.get(c, c) for c in cat_order]
sizes        = [len(groups_best[c]) for c in cat_order]

fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(
    data_ordered,
    labels=[f"{s}\n(n={n:,})" for s, n in zip(short_labels, sizes)],
    patch_artist=True,
    medianprops=dict(color="black", lw=2),
    notch=False,
)
colors = plt.cm.Set2(np.linspace(0, 1, len(cat_order)))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_ylabel("log(CTR)")
ax.set_title(
    f"log(CTR) by category — BO-optimised threshold ({BEST_THRESH:.3f})\n"
    f"One-way ANOVA: F = {BEST_F:.0f}  (p ≪ 0.001)"
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "upworthy_best_config_boxplot.png", bbox_inches="tight")
plt.show()
print("Saved: upworthy_best_config_boxplot.png")

## 7  Summary

### Key findings

| Method | Best F-statistic | Relative to BO |
|--------|-----------------|----------------|
| **Bayesian Optimisation** | **1568.7** | — |
| Random search | 1514.0 | −3.6% |
| Grid search | 1435.2 | −8.5% |

### What BO found

- **Optimal threshold ≈ 0.256** — not the liberal default (0.05) nor the conservative extreme (0.85)
- **Optimal min_cat_size ≈ 86** — removes unreliable tiny groups without discarding real categories
- **Coverage ≈ 88%** — most headlines get a label, but the least confident 12% are filtered out

### Why does the threshold matter?

At threshold=0.05 (100% coverage), every headline gets a label — even ones where BART is only
51% confident. These ambiguous headlines dilute the category signal, raising within-group variance
and lowering F.  At threshold=0.85 (3% coverage), groups become tiny and F collapses due to
insufficient data.  BO found the sweet spot where label quality and sample size are jointly optimised.

### Connection to BO theory (Lec 22–23)

- **GP surrogate**: modelled the unknown $F(\text{threshold}, \text{min\_cat})$ landscape using the SE kernel
- **EI acquisition**: balanced exploring uncertain regions (high posterior $\sigma$) with exploiting known good regions (high posterior $\mu$)
- **Convergence**: BO found the optimum by trial ≈40, spending the remaining trials confirming it
- **Advantage over grid**: grid search's 7×7=49 points never happened to land near threshold≈0.256; BO learned to look there